# 대화 메모리(Conversation Memory) 실습

LLM은 기본적으로 호출할 때마다 "이전에 무슨 대화를 했는지"를 기억하지 못한다. 그래서 채팅 기록을 직접 저장해뒀다가, 다음 호출을 할 때 그 기록을 프롬프트에 함께 넣어줘야 "이전 대화를 기억하는 것처럼" 동작하게 만들 수 있다. 이번 실습에서는 대화 기록을 저장하는 방법부터, 체인에 자동으로 연결해서 대화가 이어지도록 만드는 방법까지 실습한다.

> 참고로 [teddynote 교재의 ConversationBufferMemory 예제](https://github.com/teddylee777/langchain-kr/blob/main/05-Memory/01-ConversationBufferMemory.ipynb)는 `ConversationBufferMemory`, `ConversationChain`을 사용하는데, 이 두 클래스는 현재 LangChain 문서에서 deprecated(사용 중단)로 표시되어 있다. 여기서는 같은 개념(대화 기록 저장 → 체인에 자동 반영)을 현재 권장되는 `InMemoryChatMessageHistory` + `RunnableWithMessageHistory` 조합으로 실습한다.

## 1. 환경변수 로드

`.env`에 저장된 `OPENAI_API_KEY`를 불러온다.

In [23]:
from dotenv import load_dotenv

# .env 파일의 OPENAI_API_KEY 등 환경변수를 불러온다.
load_dotenv()

True

## 2. 대화 기록 객체 만들기

`InMemoryChatMessageHistory`는 대화 메시지들을 순서대로 저장해두는 가장 기본적인 대화 기록 저장소다. 메모리(프로세스가 실행되는 동안)에만 저장되고, 프로그램을 재시작하면 사라진다.

In [24]:
from langchain_core.chat_history import InMemoryChatMessageHistory

# 대화 메시지를 순서대로 저장해두는 객체.
history = InMemoryChatMessageHistory()
history

InMemoryChatMessageHistory(messages=[])

## 3. 대화 저장하기

`add_user_message()` / `add_ai_message()`로 사용자 발화와 AI 응답을 순서대로 저장한다. (레거시 `ConversationBufferMemory`의 `save_context(inputs, outputs)`와 같은 역할을 한다.)

In [25]:
# 사용자 발화 저장
history.add_user_message("안녕하세요, 은행에서 계좌를 개설하고 싶습니다.")
# AI 응답 저장
history.add_ai_message(
    "안녕하세요! 계좌 개설을 도와드리겠습니다. 먼저 신분증을 준비해 주시겠어요?"
)

## 4. 저장된 기록 확인하기

`history.messages`는 지금까지 저장된 메시지들을 `HumanMessage`, `AIMessage` 객체 리스트로 반환한다. (레거시 `ConversationBufferMemory`의 `load_memory_variables({})`와 같은 역할을 한다. 레거시 버전은 `return_messages=True`를 따로 지정해야 메시지 객체로 받을 수 있었지만, 여기서는 처음부터 메시지 객체로 저장/조회된다.)

In [26]:
for message in history.messages:
    # 메시지 종류(Human/AI)와 내용을 함께 출력.
    print(f"[{message.type}] {message.content}")

[human] 안녕하세요, 은행에서 계좌를 개설하고 싶습니다.
[ai] 안녕하세요! 계좌 개설을 도와드리겠습니다. 먼저 신분증을 준비해 주시겠어요?


## 5. 대화를 이어서 여러 턴 쌓기

실제 대화처럼 여러 턴을 이어서 저장해본다. 계좌 개설 절차가 진행되는 과정을 그대로 기록해봤다.

In [27]:
history.add_user_message("신분증 사진을 업로드했습니다.")
history.add_ai_message("확인했습니다. 본인 인증을 위해 휴대폰 인증을 진행해 주세요.")

history.add_user_message("휴대폰 인증을 완료했습니다.")
history.add_ai_message("인증이 완료되었습니다. 이제 계좌 종류를 선택해 주세요.")

history.add_user_message("입출금이 자유로운 보통예금 계좌로 개설해 주세요.")
history.add_ai_message("보통예금 계좌 개설 신청이 완료되었습니다. 영업일 기준 1일 이내 개설이 완료됩니다.")

for message in history.messages:
    print(f"[{message.type}] {message.content}")

[human] 안녕하세요, 은행에서 계좌를 개설하고 싶습니다.
[ai] 안녕하세요! 계좌 개설을 도와드리겠습니다. 먼저 신분증을 준비해 주시겠어요?
[human] 신분증 사진을 업로드했습니다.
[ai] 확인했습니다. 본인 인증을 위해 휴대폰 인증을 진행해 주세요.
[human] 휴대폰 인증을 완료했습니다.
[ai] 인증이 완료되었습니다. 이제 계좌 종류를 선택해 주세요.
[human] 입출금이 자유로운 보통예금 계좌로 개설해 주세요.
[ai] 보통예금 계좌 개설 신청이 완료되었습니다. 영업일 기준 1일 이내 개설이 완료됩니다.


## 6. 체인에 자동으로 연결하기: RunnableWithMessageHistory

지금까지는 메시지를 수동으로 저장/조회했다. 실제 대화형 챗봇을 만들 때는 매번 이렇게 직접 기록을 챙기는 대신, 체인 자체가 "호출할 때마다 자동으로 기록을 읽고, 새 대화를 저장"하도록 만드는 게 편하다. `RunnableWithMessageHistory`가 그 역할을 해준다. (레거시 `ConversationChain`이 하던 역할과 같다.)

먼저 대화 기록을 프롬프트에 끼워 넣을 수 있는 `ChatPromptTemplate`을 만든다. `MessagesPlaceholder("history")` 자리에 이전 대화 기록이 통째로 삽입된다.

In [28]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5.6-luna")

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 친절한 은행 상담원입니다."),
        # 이전 대화 기록이 여기에 자동으로 채워진다.
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}"),
    ]
)

chain = prompt | llm

## 7. 세션(대화)별 기록 저장소 준비하기

한 프로그램 안에서 여러 사용자가 동시에 대화할 수도 있으므로, `session_id`별로 서로 다른 대화 기록을 관리해야 한다. `session_id`를 넣으면 해당 세션의 `InMemoryChatMessageHistory`를 반환하는(없으면 새로 만드는) 함수를 정의한다.

In [29]:
# session_id -> InMemoryChatMessageHistory 를 저장해두는 딕셔너리.
store = {}


def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    # 해당 session_id의 기록이 없으면 새로 만들어서 저장.
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

## 8. RunnableWithMessageHistory로 감싸기

`chain`을 `RunnableWithMessageHistory`로 감싸면, 호출할 때마다 `get_session_history`로 해당 세션의 기록을 자동으로 불러와 `history` 자리에 채워주고, 호출이 끝나면 이번 대화(사용자 입력 + AI 응답)를 그 기록에 자동으로 저장해준다.

- `input_messages_key="input"` : 사용자 입력이 담긴 변수 이름
- `history_messages_key="history"` : 대화 기록이 채워질 `MessagesPlaceholder`의 변수 이름

In [30]:
from langchain_core.runnables.history import RunnableWithMessageHistory

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

d:\moon0902\hanwha_0902\ex0918\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


## 9. 대화 시작하기

`config`에 `session_id`를 지정해서 호출한다. 이 예시에서는 세션 이름을 `"moon"`으로 정했다.

In [31]:
config = {"configurable": {"session_id": "sam"}}

response = chain_with_history.invoke(
    {"input": "안녕하세요, 저는 샘올트먼입니다. 계좌를 개설하고 싶어요."},
    config=config,
)
print(response.content)

안녕하세요, 샘 올트먼님. 계좌 개설을 도와드리겠습니다.

다만 저는 여기서 실제 계좌를 개설할 수 없으므로, 공식 은행 앱·웹사이트 또는 영업점을 통해 진행해 주세요. 일반적으로 다음 정보와 서류가 필요합니다.

- 신분증 또는 여권
- 본인 명의 휴대전화
- 주소 및 세금 거주지
- 계좌 용도와 원하는 상품
- 경우에 따라 소득·납세자번호 등 추가 정보

먼저 **어느 국가의 은행 계좌**를 개설하려는지와 **개인계좌인지 법인계좌인지** 알려주시겠어요? 채팅에는 주민등록번호, 여권번호, 계좌 비밀번호 같은 민감한 정보는 보내지 마세요.


## 10. 이전 대화를 기억하는지 확인

같은 `session_id`("moon")로 다시 호출하면, 방금 나눈 대화가 `history`에 자동으로 반영되어 있어서 이름을 다시 알려주지 않아도 기억하고 있어야 한다.

In [32]:
response2 = chain_with_history.invoke(
    {"input": "제 이름이 뭐라고 했는지 기억하시나요?"},
    config=config,
)
print(response2.content)

네, **샘 올트먼**이라고 말씀하셨습니다.


## 11. 다른 세션은 독립적으로 관리된다

이번엔 `session_id`를 `"guest"`로 바꿔서 같은 질문을 해본다. `"moon"` 세션과는 별개의 대화 기록이므로, 이전에 이름을 알려준 적이 없어서 모른다고 답해야 한다.

In [33]:
response3 = chain_with_history.invoke(
    {"input": "제 이름이 뭐라고 했는지 기억하시나요?"},
    config={"configurable": {"session_id": "guest"}},
)
print(response3.content)

죄송하지만, 아직 성함을 말씀해 주신 기록이 없어 기억하지 못합니다. 성함을 알려주시면 앞으로 그렇게 불러드리겠습니다.


## 정리

- `InMemoryChatMessageHistory` : 대화 메시지를 순서대로 저장하는 기본 저장소 (`add_user_message`/`add_ai_message`로 저장, `.messages`로 조회)
- `RunnableWithMessageHistory` : 체인을 감싸서, 호출할 때마다 `session_id`에 해당하는 대화 기록을 자동으로 읽고 저장해준다
- `session_id`를 다르게 주면 완전히 독립된 대화로 관리된다

이 세 가지를 조합하면, 매번 직접 기록을 챙기지 않아도 "이전 대화를 기억하는" 챗봇 체인을 만들 수 있다.